In [18]:
from youtube_transcript_api import YouTubeTranscriptApi
import re
import json
from datetime import datetime

In [19]:
# YouTube URL에서 video ID 추출
def extract_video_id(url):
    match = re.search(r'(?:v=|\/)([0-9A-Za-z_-]{11})', url)
    return match.group(1) if match else None

In [20]:
# YouTube 영상에서 자막 텍스트 추출
def get_youtube_transcript(video_url):
    try:
        video_id = extract_video_id(video_url)
        if not video_id:
            return "유효하지 않은 YouTube URL입니다."
        
        api = YouTubeTranscriptApi()
        
        # 한국어 → 영어 → 자동생성 순
        for languages in [['ko'], ['en'], None]:
            try:
                transcript = api.fetch(video_id, languages=languages) if languages else api.fetch(video_id)
                # 텍스트 추출
                if hasattr(transcript, 'to_raw_data'):
                    data = transcript.to_raw_data()
                else:
                    data = transcript
                return " ".join([item['text'] for item in data])
            except:
                continue
        
        return "자막을 찾을 수 없습니다."
    
    except Exception as e:
        return f"에러: {str(e)}"

In [21]:
# 타임스탬프 포함 자막 추출
def get_transcript_with_timestamps(video_url):
    try:
        video_id = extract_video_id(video_url)
        if not video_id:
            return []
        
        api = YouTubeTranscriptApi()
        
        for languages in [['ko'], ['en'], None]:
            try:
                transcript = api.fetch(video_id, languages=languages) if languages else api.fetch(video_id)
                return transcript.to_raw_data() if hasattr(transcript, 'to_raw_data') else transcript
            except:
                continue
        
        return []
    
    except Exception as e:
        return []

In [ ]:
# JSON 파일로 저장
def save_transcript_to_json(video_url):
    try:
        video_id = extract_video_id(video_url)
        if not video_id:
            return {"error": "유효하지 않은 YouTube URL입니다."}
        
        # 자막 데이터 가져오기
        transcript_data = get_transcript_with_timestamps(video_url)
        text_only = get_youtube_transcript(video_url)
        
        if not transcript_data:
            return {"error": "자막을 가져올 수 없습니다."}
        
        # JSON 데이터 구성
        json_data = {
            "video_id": video_id,
            "video_url": video_url,
            "extracted_at": datetime.now().isoformat(),
            "text_only": text_only,
            "segments": transcript_data,
            "total_segments": len(transcript_data)
        }
        
        # 기본 파일명으로 저장
        output_path = f"transcript_{video_id}.json"
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)
        
        return {"success": True, "file_path": output_path}
    
    except Exception as e:
        return {"error": f"저장 실패: {str(e)}"}

In [29]:
# 테스트
if __name__ == "__main__":
    url = "https://www.youtube.com/shorts/vkAKkeVNj_A"
    
    # 텍스트만 추출
    text = get_youtube_transcript(url)
    print(f"자막 텍스트: {text[:100]}...")
    
    # 타임스탬프 포함
    timestamps = get_transcript_with_timestamps(url)
    if timestamps:
        print(f"첫 번째 세그먼트: {timestamps[0]}")

    # JSON 파일로 저장
    result = save_transcript_to_json(url)
    if result.get("success"):
        print(f"✓ JSON 파일 저장 완료: {result['file_path']}")
    else:
        print(f"✗ 저장 실패: {result.get('error')}")

자막 텍스트: 그 펌프는 있으니까 자바라는 있으니까 그 오일통은 [음악] [웃음]...
첫 번째 세그먼트: {'text': '그 펌프는 있으니까 자바라는 있으니까', 'start': 0.06, 'duration': 5.0}
✓ JSON 파일 저장 완료: youtube_caption/transcript_vkAKkeVNj_A.json
